> `oeai_mod_wonde_env_var.ipynb`
> [20251105.1]
> *Module configuration*

In [0]:
%run ./oeai_py

In [0]:
%run ./oeai_logger

In [0]:
# Create an instance of OEAI class and set the plaform ("Azure", "Fabric"...)
oeai = OEAI(platform="Fabric")

In [0]:
# CHANGE VALUES FOR YOUR KEY VAULT
keyvault = "" # Fabric requires full URL eg "https://key_vault_name.vault.azure.net/"
keyvault_linked_service = "" # Not required for Fabric.

In [0]:
# INITIALISE LOGGING
oeai.log = OEAILogger()
oeai.module_id = "wonde"

# SET HANDLER - Console Output
oeai.log.handler.stream = SimpleNamespace(
    level=_lib_logging.INFO, 
    formatter="stream"
)

In [0]:
# OEA environment paths
import os
from types import SimpleNamespace
storage_root = oeai.get_secret(spark, "storage-root", keyvault_linked_service, keyvault)
oeai.path = SimpleNamespace(
    reference = os.path.join(storage_root, f"reference/"),
    bronze    = os.path.join(storage_root, f"oeai_bronze/{oeai.module_id}/"),
    silver    = os.path.join(storage_root, f"oeai_silver/"),
    gold      = os.path.join(storage_root, f"oeai_gold/"),
)

bronze_path = oeai.path.bronze
silver_path = oeai.path.silver
gold_path   = oeai.path.gold

In [0]:
# Wonde: School ID list
school_ids = oeai.get_secret(spark, "wonde-school-ids", keyvault_linked_service, keyvault).split(",")

In [0]:
from datetime import datetime

def get_ac_year(date=None):
    """
    Return the academic year for the given date.
    
    Academic year is the calendar year if month >= August, else the previous year.
    
    Args:
        date (str | datetime.datetime | None): 
            - If None, uses today.
            - If str, parsed with dateutil.parser.
            - If datetime, used directly.
    
    Returns:
        int: Academic year.
    
    Raises:
        ValueError: if string cannot be parsed as a date.
        TypeError: if date is not str or datetime.
    """
    # from datetime import datetime
    try:
        from dateutil import parser
    except ImportError:
        raise ImportError(
            "Please install python-dateutil to parse date strings: pip install python-dateutil"
        )

    # default to now
    if date is None:
        dt = datetime.now()
    # parse strings
    elif isinstance(date, str):
        try:
            dt = parser.parse(date, dayfirst=True)
        except (ValueError, TypeError) as e:
            raise ValueError(f"Could not parse date string {date!r}: {e}")
    # already datetime?
    elif isinstance(date, datetime):
        dt = date
    else:
        raise TypeError(f"date must be str or datetime, got {type(date).__name__!r}")

    return dt.year if dt.month >= 8 else dt.year - 1

In [0]:
### BRONZE ###
# Environment notebook for job definitions and overrides in Bronze Notebook

# Daily jobs
daily_jobs = [
    ("", "schools", "cursor", None, "", False, False), 
    ("/attendance-codes", "attendance_codes", "cursor", None, "", False, False),
    ("/attendance/session", "attendance_session", "cursor", None, "", True, True),
    ("/attendance-leavers/session", "attendance_leavers_session", "cursor", None, "", True, True),
    ("/behaviours", "behaviours_students", "cursor", None, "&include=students", True, True),
    ("/achievements", "achievements_students", "cursor", None, "&include=students", True, True),
    ("/exclusions", "exclusions", "cursor", None, "&include=student", True, False),
    ("/exclusions", "exclusions_leaver", "cursor", None, "&include=student-leaver", True, False), 
    ("/groups", "groups", "cursor", None, "", False, False),
    #("/assessment/aspects", "aspects", "cursor", None, "", False, True),
    #("/assessment/results", "results", "cursor", None, "", False, True),
    #("/assessment/resultsets", "resultsets", "cursor", None, "", False, True),
    ("/deletions", "deletions_achievement", "cursor", None, "type=achievement", True, False), 
    ("/deletions", "deletions_behaviour", "cursor", None, "type=behaviour", True, False), 
    ("/deletions", "deletions_exclusion", "cursor", None, "type=exclusion", True, False), 
    ("/deletions", "deletions_student", "cursor", None, "type=student", True, False),
    ("/employee-absences", "employee_absences", "cursor", None, "", True, False),
    ("/lessons", "lessons", "cursor", None, "&include=class", False, False),
    ("/attendance/lesson", "attendance_lesson", "cursor", None, "", True, True),
    ("/photos", "photos", "cursor", None, "", True, False),
    ("/events", "events", "cursor", None, "", False, False),
]

# Student jobs (includes both students and leavers)
student_jobs = [
    ("/students", "students", "cursor", None, "", False, False), 
    ("/students", "students_education", "cursor", None, "&include=education_details", False, False), 
    ("/students", "students_extended", "cursor", None, "&include=extended_details", False, False), 
    ("/students", "students_contact_details", "cursor", None, "&include=contact_details", False, False),
    ("/students", "students_sen_needs", "cursor", None, "&include=sen_needs", False, True),
    ("/students", "students_udf", "cursor", None, "&include=user_defined_fields", False, True),
    ("/students-pre-admission", "students_pre_admission", "cursor", None, "", False, False), 
    ("/students", "students_groups", "cursor", None, "&include=groups", False, False),
    ("/students-leaver", "students_leaver", "cursor", None, "", False, False), 
    ("/students-leaver", "students_leaver_education", "cursor", None, "&include=education_details", False, False), 
    ("/students-leaver", "students_leaver_extended", "cursor", None, "&include=extended_details", False, False), 
    ("/students-leaver", "students_leaver", "cursor", None, "&include=education_details", False, False), 
    ("/classes", "classes", "cursor", None, "include=subject", False, False),
    ("/students-leaver", "students_leaver_cbds", "cursor", None, "&include=cbds", False, False),
]

# Staff jobs
staff_jobs = [
    ("/employees", "employees", "cursor", None, "", False, False), 
    ("/employees", "employees_detail", "cursor", None, "&include=employment_details", False, False), 
]

# Overrides for specific job types
job_overrides = {
    "daily_jobs": f"",  # Daily job override date - leave blank "" to use last updated or use formatted date eg: 2021-08-01 00:00:01 
    "students"  : f"{get_ac_year()-9}-08-01 00:00:01",    # One override for both students and students-leaver
    "employees" : f"{get_ac_year()-9}-08-01 00:00:01",    # Staff jobs override
}

### SILVER ###
# Define the mapping between JSON files and desired Delta table names in Silver Notebook

delta_table_name_mapping = {
    "achievements_students.json": "fact_Achievement",
    "attendance_codes.json": "",
    "attendance_lesson.json": "fact_AttendanceLesson",
    "attendance_leavers_session.json": "",
    "attendance_session.json": "fact_AttendanceSession",
    "behaviours_students.json": "fact_Behaviour",
    "classes.json": "",
    "deletions_achievement.json": "fact_Deletion",
    "deletions_behaviour.json": "fact_Deletion",
    "deletions_exclusion.json": "fact_Deletion",
    "deletions_student.json": "fact_Deletion",
    "deletions_employee.json": "fact_Deletion",            
    "deletions_employee_absence.json": "fact_Deletion",    
    "employee_absences.json": "fact_StaffAbsence",
    "employees.json": "dim_Staff",
    "employees_detail.json": "",
    "events.json": "dim_Event",
    "exclusions.json": "fact_Exclusion",
    "exclusions_leaver.json": "",
    "groups.json": "dim_Group",
    "schools.json": "dim_Organisation",
    "students.json": "dim_Student",
    "students_contact_details.json": "dim_Address",
    "students_education.json": "",
    "students_extended.json": "dim_StudentExtended",
    "students_groups.json": "dim_GroupMembership",
    "students_leaver.json": "",
    "students_leaver_cbds.json": "",
    "students_leaver_education.json": "",
    "students_leaver_extended.json": "",
    "students_pre_admission.json": "",
    "students_sen_needs.json": "dim_SENDNeed",
    "students_udf.json": "dim_UserDefinedFields",
    "exclusions_reason.json": "dim_ExclusionReason",
    
    # "aspects.json": "",
    # "results.json": "fact_Attainment",
    # "resultsets.json": "",
    # "lessons.json": "",
    # "subjects.json": "",
    # "periods.json": "",
    # "attendance_detention.json": "fact_AttendanceDetention",
    # "detention-attendance_codes.json": "",
    # "employees_roles.json": "dim_StaffContractual",
    # "employees_salary.json": "dim_StaffSalary",
    # "employees_allowance.json": "",
    # "employees_scale.json": "dim_StaffScale",
    # "employees_qualifications.json": "dim_StaffQualification",
    # "employees_checks.json": "dim_StaffEmploymentCheck",
}

In [0]:
# Use parameters in pipeline to customise run
try:
    if custom_run == "refresh":
        # Use defaults
        pass

    if custom_run == "rebase":
        daily_jobs = [
            job for job in daily_jobs
            if "deletion" not in job[1]
        ]

        job_overrides = {
            "daily_jobs": f"{get_ac_year()-3}-08-01 00:00:01",  # Daily job override date - leave blank "" to use last updated or use formatted date eg: 2021-08-01 00:00:01 
            "students"  : f"{get_ac_year()-9}-08-01 00:00:01",    # One override for both students and students-leaver
            "employees" : f"{get_ac_year()-9}-08-01 00:00:01",    # Staff jobs override
        }
    
    if custom_run == "exclude_ach_beh":
        daily_jobs = [
            job for job in daily_jobs
            if "achievement" not in job[1] and "behaviour" not in job[1]
        ]
    
    if custom_run == "only_achievements":
        daily_jobs = [
            job for job in daily_jobs
            if "achievement"in job[1]
        ]
        # Student jobs = None/Skip
        student_jobs = []
        # Staff jobs = None/Skip
        staff_jobs = []
    
    if custom_run == "only_behaviours":
        daily_jobs = [
            job for job in daily_jobs
            if "behaviour"in job[1]
        ]
        # Student jobs = None/Skip
        student_jobs = []
        # Staff jobs = None/Skip
        staff_jobs = []
    
    if custom_run == "only_attendance":
        daily_jobs = [
            ("/attendance/session", "attendance_session", "cursor", None, "", True, True),
        ]
        # Student jobs = None/Skip
        student_jobs = []
        # Staff jobs = None/Skip
        staff_jobs = []

        # Overrides for specific job types
        job_overrides = {
            "daily_jobs": f"{datetime.now():%Y-%m-%d} 00:00:01",  # Daily job override date = Today
            "students"  : f"{get_ac_year()-9}-08-01 00:00:01",
            "employees" : f"{get_ac_year()-9}-08-01 00:00:01",
        }
   
except NameError:
    print("custom_run is not defined, proceeding with Default Environment Variables.")

else:
    print(f"Custom Environment Notebook Loaded Successfully [{custom_run}]")